# Phase 8 — ADAM Saddle-Aware Energy Minimization

Soft-particle landscapes at high $\phi$ are **saddle-rich**: each contact has azimuthal sliding freedom and the deformable shape DOFs ($2 \cdot P \cdot N$) far outnumber contact-normal constraints. Isotropic damping (xi-drag, $\beta_{rb}$) is direction-blind and traps the system at the first saddle encountered.

**ML-style adaptive optimizers** (ADAM, AdamW, Lion) escape saddles via per-DOF momentum + adaptive learning rate — effectively per-coordinate Hessian preconditioning. They are the right tool for IC preparation in dense soft-particle systems where mechanical equilibrium is glassy/marginal.

**This notebook demonstrates the full Phase 8 workflow:**
1. Build a 16-particle periodic system, init to $\phi = 0.49$
2. Adaptive box-rate swell to $\phi = 0.86$ (annealing oscillation near target)
3. Snapshot baseline: $PE_{\rm swell}$ with per-term breakdown
4. **ADAM optimization** for 5000 steps (xi=alpha=$\beta_{rb}$=0)
5. Snapshot $PE_{\rm adam}$ with breakdown
6. Negative control: pure-Newtonian physics from swell state for the same wall time
7. Convergence plots: $PE$ vs step **and $|F|$ vs step** (force-balance quality)

Phase 8 is **purely additive** — no changes to the existing kernel or candidacy code.
Four new methods on `System`: `make_position_variable`, `eval_forces_at`, `eval_potential_energy[_components]`, `sync_from_var`.

---
## Section 0 — Setup (Colab)

Run once per session. Clones the repo, installs deps, builds the C++ candidacy extension.

In [ ]:
import os, sys

REPO_URL = 'https://github.com/KD-physics/Squishy-Particle-Simulator.git'

if not os.path.exists('Squishy-Particle-Simulator'):
    !git clone {REPO_URL}
%cd Squishy-Particle-Simulator/python_v2

!pip install -q numpy scipy matplotlib imageio tensorflow
!pip install -q .    # builds C++ candidacy-manager extension

print('Setup complete.')

---
## Section 1 — Imports + system build

In [ ]:
import time
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from IPython.display import display

import src.simulation.tf_sim as tfm
tfm.set_dtype(tf.float64)   # always use float64; float32 loses contact accuracy

from src.epd.particles import ParticleSpec
from src.epd.system import System

# 16-particle periodic emulsion system, bimodal radii
P, N        = 16, 60
PHI_INIT    = 0.49     # post-jamming — standard isostatic starting point
PHI_TARGET  = 0.86     # well into the high-φ regime where ADAM matters
OH          = 5.0      # physical Ohnesorge for soap-stabilised W/O emulsion
SEED        = 42

s = System(Lx=18.0, Ly=18.0, periodic_x=True, periodic_y=True,
           dt_factor=0.25, candidacy_kind='prcm', E_candidates=9)
s.add_particles(ParticleSpec(
    count=P, type='emulsion', gamma=1.0, kappa=0.02,
    N_nodes=N, Oh=OH,
    poly_dist={'type': 'bimodal', 'ratio': 0.5, 'delta': 0.2}))

# Standard adaptive_swell to phi=0.49 (just above isostatic jamming)
s.initialize(phi_target=PHI_INIT, seed=SEED, verbose=False,
             relax_only=False, n_relax_init=200)

DTYPE = s._params['_alpha_tf'].dtype
dt    = float(s._params['_dt_tf'].numpy())
print(f'phi_after_init = {s.phi_outer:.4f}   Lx = {s.Lx:.4f}   dt = {dt:.4e}')

# Save the physical xi for the regression check at the end
xi_phys = tf.constant(s._params['xi_drag_per_p'].numpy(),
                       dtype=s._params['xi_drag_per_p'].dtype)

---
## Section 2 — Adaptive box-rate swell to $\phi = 0.86$

**Strategy** (matches `src/epd/tests/test_phase8.py`):
1. Per-chunk target rate from local $\dot\phi$: `rate_test = -((φ_tgt - φ) / (n_remain·dt)) / (2·φ)`
2. **Monotonically non-increasing rate**: adopt `rate_test` only if smaller; else decay current by 1%
3. **Annealing oscillation** when within 0.02 of target: alternate compress×2 / expand×0.5 to relieve contact stress without overshoot
4. $\beta_{rb} = 0.005/dt$ rigid-body damping + per-chunk mass-weighted drift removal
5. $\alpha \cdot dt = 0.2$ internal-mode damping (semi-implicit, unconditionally stable)
6. Seed initial rate at **2 × rate_test** to avoid the decay-only death spiral

In [ ]:
N_SWELL  = 100_000
CHUNK    = max(N_SWELL // 125, 100)
phi_init = s.phi_outer

# Damping during swell
SWELL_ALPHA = 0.2 / dt
s._params['_alpha_tf']        = tf.constant(SWELL_ALPHA, dtype=DTYPE)
s._params['alpha_damp_per_p'] = tf.constant(np.full(P, SWELL_ALPHA), dtype=DTYPE)
s.beta_rb = 0.005 / dt

# Seed initial rate: 2× the rate that would (linearly) reach target in N_SWELL steps
target_dphi_dt_init = (PHI_TARGET - phi_init) / (N_SWELL * dt)
rate_init           = -2.0 * target_dphi_dt_init / (2.0 * phi_init)
s.box_rate          = (rate_init, rate_init)

oscillation_triggered = False
direction = -1.0  # -1 = compressing
N_SWELL_CAP = int(1.5 * N_SWELL)

steps = 0
t0 = time.time()
phi_history = [(0, s.phi_outer)]
while s.phi_outer < PHI_TARGET - 1e-6 and steps < N_SWELL_CAP:
    phi_now  = s.phi_outer
    n_remain = max(N_SWELL - steps, CHUNK)
    target_dphi_dt = (PHI_TARGET - phi_now) / (n_remain * dt)
    rate_test = -target_dphi_dt / (2.0 * phi_now)
    cur_rate  = s.box_rate[0]
    if abs(rate_test) < abs(cur_rate) and steps < N_SWELL:
        rate = rate_test
    else:
        rate = cur_rate * 0.99

    if (PHI_TARGET - phi_now) <= 0.02 or oscillation_triggered:
        if not oscillation_triggered:
            oscillation_triggered = True
            print(f'  [anneal] oscillation triggered at Δφ = {PHI_TARGET - phi_now:.4f}  (steps={steps})')
        if direction < 0:
            rate =  abs(rate) / 2
        else:
            rate = -abs(rate) * 2
        direction = -direction

    s.box_rate = (rate, rate)
    s.run(CHUNK, sample_every=CHUNK, verbose=False, record_initial=False)
    s.sync_box(CHUNK)
    steps += CHUNK

    # Per-chunk mass-weighted drift removal (periodic-system best practice)
    M_np = s._params['M_disk'].numpy()
    I_np = s._params['I_disk'].numpy()
    v_cm_np  = s._state['v_cm'].numpy()
    omega_np = s._state['omega'].numpy()
    v_mean    = (M_np[:, None] * v_cm_np).sum(axis=0) / M_np.sum()
    omega_mean = (I_np * omega_np).sum() / I_np.sum()
    s._state['v_cm']  = tf.constant(v_cm_np  - v_mean,    dtype=DTYPE)
    s._state['omega'] = tf.constant(omega_np - omega_mean, dtype=DTYPE)
    phi_history.append((steps, s.phi_outer))

s.box_rate = 0.0
swell_wall = time.time() - t0
print(f'\nSwell done: {steps} steps in {swell_wall:.1f}s  →  '
      f'φ = {s.phi_outer:.4f}  (target {PHI_TARGET:.4f})  Lx = {s.Lx:.4f}')

# Plot φ vs steps
fig, ax = plt.subplots(figsize=(8, 4))
steps_arr, phi_arr = zip(*phi_history)
ax.plot(steps_arr, phi_arr, 'b-', lw=1.5)
ax.axhline(PHI_TARGET, color='k', ls='--', label=f'target φ = {PHI_TARGET}')
ax.set_xlabel('swell step')
ax.set_ylabel('φ_outer')
ax.legend(); ax.grid(alpha=0.3); ax.set_title('Adaptive box-rate swell')
plt.show()

---
## Section 3 — Baseline $PE_{\rm swell}$

Zero **all** damping (xi, alpha, $\beta_{rb}$) so both ADAM and the Newtonian negative
control share the same baseline. Capture per-term PE breakdown for the diagnostic table.

In [ ]:
s._params['xi_drag_per_p']    = tf.zeros_like(xi_phys)
s._params['_alpha_tf']        = tf.constant(0.0, dtype=DTYPE)
s._params['alpha_damp_per_p'] = tf.zeros([P], dtype=DTYPE)
s.beta_rb = 0.0

PE_swell      = s.eval_potential_energy()
PE_swell_comp = s.eval_potential_energy_components()
print(f'PE_swell = {PE_swell:.4e}')
for k in ['edge', 'bend', 'area', 'lt', 'cc', 'prim', 'total']:
    print(f'    {k:<6} = {PE_swell_comp[k]:.4e}')

# Snapshot the swell configuration for side-by-side comparison
s.set_color_palette('palette1', seed=1)
fig, ax = plt.subplots(figsize=(7, 7))
s.render(ax=ax)
ax.set_title(f'Swell baseline  φ={s.phi_outer:.4f}  PE={PE_swell:.2e}')
plt.show()

---
## Section 4 — ADAM saddle-quench (5000 steps)

Pure-TF ADAM optimization on $x_{\rm all}$ as the free parameter, with `eval_forces_at`
providing the gradient. **State of the System is NOT mutated** during ADAM — the
optimizer operates on a separate `tf.Variable`.

We sample two diagnostics each `SAMPLE_EVERY` steps:
- $PE(x_{\rm var})$ — total potential energy (overall progress)
- $|F|_{\max}, |F|_{\rm mean}$ — per-DOF force imbalance (closer to mechanical
  equilibrium $\Leftrightarrow$ smaller force residual). At a true minimum, $|F| \to 0$.

In [ ]:
N_ADAM     = 5000
ADAM_LR    = 1e-3
PRCM_EVERY = 50
SAMPLE_EVERY = max(1, N_ADAM // 25)

opt   = tf.keras.optimizers.Adam(learning_rate=ADAM_LR)
x_var = s.make_position_variable()

# Refresh CapCandidates to align with x_var (defensive)
x_np    = x_var.numpy(); x_cm_np = x_np.mean(axis=1)
theta_np = s._state['theta'].numpy()
s._cm_mgr.update(x_cm_np, theta_np, x_all=x_np)

# Initial baseline force-balance reading (before any ADAM step)
f0      = s.eval_forces_at(x_var)
f0_mag  = tf.norm(f0, axis=2).numpy()
print(f'Initial |F|: max={f0_mag.max():.3e}  mean={f0_mag.mean():.3e}')

t0 = time.time()
adam_log = []   # (step, PE, max|F|, mean|F|)
# Record the initial point as step 0
adam_log.append((0, PE_swell, float(f0_mag.max()), float(f0_mag.mean())))
for i in range(N_ADAM):
    f = s.eval_forces_at(x_var)              # tape-free, JIT-compiled
    opt.apply_gradients([(-f, x_var)])       # ADAM treats -f as gradient
    if (i + 1) % PRCM_EVERY == 0:
        x_np = x_var.numpy(); x_cm_np = x_np.mean(axis=1)
        if s._cm_mgr.needs_update(x_cm_np, theta_np):
            s._cm_mgr.update(x_cm_np, theta_np, x_all=x_np)
    if (i + 1) % SAMPLE_EVERY == 0:
        x_now  = tf.constant(x_var.numpy(), dtype=DTYPE)
        PE_now = s.eval_potential_energy(x_now)
        f_now  = s.eval_forces_at(x_var)
        f_mag  = tf.norm(f_now, axis=2).numpy()
        adam_log.append((i + 1, PE_now, float(f_mag.max()), float(f_mag.mean())))
adam_wall = time.time() - t0

PE_adam      = s.eval_potential_energy(tf.constant(x_var.numpy(), dtype=DTYPE))
PE_adam_comp = s.eval_potential_energy_components(tf.constant(x_var.numpy(), dtype=DTYPE))
f_final      = s.eval_forces_at(x_var)
f_final_mag  = tf.norm(f_final, axis=2).numpy()
print(f'\nADAM {N_ADAM} steps in {adam_wall:.2f}s  ({adam_wall/N_ADAM*1000:.2f} ms/step)')
print(f'PE_adam = {PE_adam:.4e}  (drop = {PE_swell - PE_adam:.3e}, '
      f'{(1-PE_adam/PE_swell)*100:.1f}% reduction)')
print(f'Final |F|: max={f_final_mag.max():.3e}  mean={f_final_mag.mean():.3e}  '
      f'(initial: max={f0_mag.max():.3e}, mean={f0_mag.mean():.3e})')
for k in ['edge', 'bend', 'area', 'lt', 'cc', 'prim', 'total']:
    print(f'    {k:<6} = {PE_adam_comp[k]:.4e}')

---
## Section 4b — Langevin-augmented ADAM (σ-sweep)

Does adding stochastic position noise help ADAM escape the plateau? **Theory:** at a saddle, gradient descent stalls; injecting Gaussian noise in $x$ (Langevin dynamics) gives random kicks that may carry the system over saddle barriers.

**Safety constraint** — noise step magnitude must be much smaller than the contact radius $r_c \approx L_0 \approx 2\pi/N$ (here ~0.105) or contact penalty forces blow up. ADAM's per-step displacement is ~1e-3, so we sweep $\sigma \in \{0, 10^{-4}, 3 \times 10^{-4}, 10^{-3}\}$ to keep 3-σ kicks ≤ 3e-3 (≪ r_c).

We snapshot the swell state, then run all four optimizers from the *same* starting point so the comparison is fair.

In [ ]:
# NOTE: Section 4 ran ADAM and mutated the PRCM but did NOT touch s._state
# (we only sync_from_var'd in Section 6 below). However, Section 5 (Newtonian
# control) DID mutate s._state. So we re-snapshot the un-mutated state here.
# If you skipped Section 5, the snapshot below still works.
swell_x_all = s._state['x_all'].numpy().copy()
swell_x_cm  = s._state['x_cm'].numpy().copy()
swell_theta = s._state['theta'].numpy().copy()
swell_u     = s._state['u'].numpy().copy()
swell_X_ref = s._state['X_ref'].numpy().copy()

def restore_swell():
    """Reset s._state to the post-swell snapshot (no save_state/restore_state
    so PRCM rebuild non-determinism doesn't shift the PE baseline)."""
    s._state['x_all'] = tf.constant(swell_x_all, dtype=DTYPE)
    s._state['x_cm']  = tf.constant(swell_x_cm,  dtype=DTYPE)
    s._state['theta'] = tf.constant(swell_theta, dtype=DTYPE)
    s._state['u']     = tf.constant(swell_u,     dtype=DTYPE)
    s._state['X_ref'] = tf.constant(swell_X_ref, dtype=DTYPE)
    s._state['v_cm']  = tf.zeros_like(s._state['v_cm'])
    s._state['omega'] = tf.zeros_like(s._state['omega'])
    s._state['u_dot'] = tf.zeros_like(s._state['u_dot'])
    s._cm_mgr.update(swell_x_cm, swell_theta, x_all=swell_x_all)

SIGMAS = [0.0, 1e-4, 3e-4, 1e-3]   # 0 = pure ADAM (control)
N_OPT  = 5000
SAMPLE_EVERY_LV = max(1, N_OPT // 25)
PRCM_EVERY_LV   = 50

results_lv = {}
for sigma in SIGMAS:
    restore_swell()
    opt   = tf.keras.optimizers.Adam(learning_rate=ADAM_LR)
    x_var = s.make_position_variable()
    theta_np = s._state['theta'].numpy()
    s._cm_mgr.update(x_var.numpy().mean(axis=1), theta_np, x_all=x_var.numpy())

    f0       = s.eval_forces_at(x_var)
    f0_mag   = tf.norm(f0, axis=2).numpy()
    log      = [(0, PE_swell, float(f0_mag.max()), float(f0_mag.mean()))]
    sigma_tf = tf.constant(sigma, dtype=DTYPE)
    label    = f'σ={sigma:.0e}' if sigma > 0 else 'pure ADAM'
    print(f'  Running {label}...', end=' ', flush=True)
    t0 = time.time()
    for i in range(N_OPT):
        f = s.eval_forces_at(x_var)
        opt.apply_gradients([(-f, x_var)])
        if sigma > 0:
            x_var.assign_add(tf.random.normal(tf.shape(x_var),
                                               stddev=sigma_tf, dtype=DTYPE))
        if (i + 1) % PRCM_EVERY_LV == 0:
            x_np = x_var.numpy()
            if s._cm_mgr.needs_update(x_np.mean(axis=1), theta_np):
                s._cm_mgr.update(x_np.mean(axis=1), theta_np, x_all=x_np)
        if (i + 1) % SAMPLE_EVERY_LV == 0:
            x_now  = tf.constant(x_var.numpy(), dtype=DTYPE)
            PE_now = s.eval_potential_energy(x_now)
            f_now  = s.eval_forces_at(x_var)
            f_mag  = tf.norm(f_now, axis=2).numpy()
            log.append((i + 1, PE_now, float(f_mag.max()), float(f_mag.mean())))
    results_lv[sigma] = {'log': log, 'wall': time.time() - t0}
    print(f'done in {results_lv[sigma]["wall"]:.1f}s  '
          f'PE={log[-1][1]:.4e}  |F|_max={log[-1][2]:.3e}  '
          f'|F|_mean={log[-1][3]:.3e}')

# Summary
print('\n  method           PE_final     |F|_max     |F|_mean')
print('  ' + '-' * 56)
for sigma in SIGMAS:
    log = results_lv[sigma]['log']
    label = f'σ={sigma:.0e}' if sigma > 0 else 'pure ADAM'
    print(f'  {label:<14}  {log[-1][1]:>10.4e}  {log[-1][2]:>10.3e}  {log[-1][3]:>10.3e}')

In [ ]:
# σ-sweep convergence plot — PE and |F| for each sigma
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
colors = {0.0: 'k', 1e-4: 'b', 3e-4: 'g', 1e-3: 'r'}
labels = {0.0: 'pure ADAM', 1e-4: 'σ=1e-4', 3e-4: 'σ=3e-4', 1e-3: 'σ=1e-3'}

# PE
for ax, ylog in zip(axes[0], [False, True]):
    for sigma in SIGMAS:
        log = results_lv[sigma]['log']
        its = [r[0] for r in log]; pes = [r[1] for r in log]
        ax.plot(its, pes, '-o', ms=3, c=colors[sigma], label=labels[sigma], alpha=0.85)
    ax.set_xlabel('optimizer step')
    ax.set_ylabel('PE')
    if ylog:
        ax.set_yscale('log')
    ax.legend(); ax.grid(alpha=0.3)
    ax.set_title(f"PE convergence  ({'log' if ylog else 'linear'} y)")

# |F|
for ax, ylog in zip(axes[1], [False, True]):
    for sigma in SIGMAS:
        log = results_lv[sigma]['log']
        its = [r[0] for r in log]
        f_max  = [r[2] for r in log]
        f_mean = [r[3] for r in log]
        ax.plot(its, f_max,  '-o', ms=3, c=colors[sigma],
                label=f'{labels[sigma]} |F|_max', alpha=0.85)
        ax.plot(its, f_mean, '--s', ms=3, c=colors[sigma],
                label=f'{labels[sigma]} |F|_mean', alpha=0.6)
    ax.set_xlabel('optimizer step')
    ax.set_ylabel('|F| per node')
    if ylog:
        ax.set_yscale('log')
    ax.legend(fontsize=7, ncol=2); ax.grid(alpha=0.3)
    ax.set_title(f"Force-balance convergence  ({'log' if ylog else 'linear'} y)")

plt.tight_layout()
plt.show()

**Interpretation of the σ-sweep result.**

In standalone tests (16-particle system, 5000 steps each), Langevin noise at safe σ values (kicks ≪ r_c) **does not improve** convergence. Both PE and |F| floors *rise* monotonically with σ:

| method | PE_final | \|F\|_max | \|F\|_mean |
|--------|----------|-----------|------------|
| pure ADAM | 105.60 | 0.561 | 0.0149 |
| σ=1e-4 | 105.65 | 1.389 | 0.033 |
| σ=3e-4 | 105.71 | 2.311 | 0.049 |
| σ=1e-3 | 106.01 | 3.588 | 0.093 |

**Why this is informative**: it tells us ADAM's plateau at PE=105.6 / |F|_mean=0.015 is *not* a saddle ADAM can't escape — it's a genuine local minimum (or a saddle whose escape direction has barrier larger than safe-σ kicks can carry). The |F| floor rises ∝ σ because each random kick contributes a standing-wave residual; PE rises because random kicks usually push *away* from the local optimum, not toward a deeper one.

**To go deeper from here would need:**
- Simulated annealing (start large σ, decay to 0) — exploration first, settle later
- True saddle-aware methods (cubic regularization, FIRE) — Hessian-informed step direction
- Smaller lr with longer runs — resolves floor better but doesn't escape basins

The noise budget that *would* actually escape (σ approaching r_c) blows up contact penalty forces. So unmodified Langevin isn't the right tool here — PE=105.6 is the basin the saddle-rich landscape allows ADAM to find.

---
## Section 5 — Negative control: pure Newtonian physics

Run the same number of steps from the swell state with **all damping = 0**, so PE+KE
is conserved. PE oscillates around $PE_{\rm swell}$ but doesn't drop monotonically.
If $PE_{\rm adam} \ll PE_{\rm physics}$, ADAM is doing real saddle-escape work.

In [ ]:
N_CTRL = N_ADAM   # same step budget for fair comparison

# Refresh CapCandidates for the original swell-state positions (s._state was un-mutated)
s._cm_mgr.update(s._state['x_cm'].numpy(),
                  s._state['theta'].numpy(),
                  x_all=s._state['x_all'].numpy())

t0 = time.time()
s.run(N_CTRL, sample_every=N_CTRL, verbose=False, record_initial=False)
ctrl_wall = time.time() - t0

PE_physics      = s.eval_potential_energy()
PE_physics_comp = s.eval_potential_energy_components()
print(f'Newtonian control: {N_CTRL} steps in {ctrl_wall:.2f}s  '
      f'({ctrl_wall/N_CTRL*1000:.2f} ms/step)')
print(f'PE_physics = {PE_physics:.4e}')
for k in ['edge', 'bend', 'area', 'lt', 'cc', 'prim', 'total']:
    print(f'    {k:<6} = {PE_physics_comp[k]:.4e}')

---
## Section 6 — Side-by-side comparison

Three states: swell (compressed, un-relaxed) → ADAM (saddle-escaped) → Newtonian (drift).

In [ ]:
# Per-term comparison table
print(f'  {"term":<6}  {"swell":>11}  {"adam":>11}  {"physics":>11}  {"Δ(adam-swell)":>13}')
print(f'  {"-"*6}  {"-"*11}  {"-"*11}  {"-"*11}  {"-"*13}')
for k in ['edge', 'bend', 'area', 'lt', 'cc', 'prim', 'total']:
    s_v = PE_swell_comp[k]; a_v = PE_adam_comp[k]; p_v = PE_physics_comp[k]
    print(f'  {k:<6}  {s_v:>11.4e}  {a_v:>11.4e}  {p_v:>11.4e}  {a_v-s_v:>+13.4e}')

# Render: (a) Newtonian-control state — currently in s._state
# (b) ADAM state — sync_from_var commits x_var into s._state
fig, axes = plt.subplots(1, 3, figsize=(20, 7))

s.set_color_palette('palette1', seed=1)
s.render(ax=axes[2])
axes[2].set_title(f'Newtonian control\nPE={PE_physics:.3e}  φ={s.phi_outer:.4f}')

s.sync_from_var(x_var)
s.set_color_palette('palette1', seed=1)
s.render(ax=axes[1])
axes[1].set_title(f'After ADAM ({N_ADAM} steps)\nPE={PE_adam:.3e}  φ={s.phi_outer:.4f}')

axes[0].axis('off')
axes[0].text(0.5, 0.55,
             f'Swell baseline\nPE={PE_swell:.3e}\nφ={PHI_TARGET:.4f}\n\n'
             f'(rendered above as Section 3)',
             ha='center', va='center', fontsize=14, transform=axes[0].transAxes)

plt.tight_layout()
plt.show()

---
## Section 7 — Convergence: $PE$ and $|F|$ vs ADAM step

Two diagnostics tell complementary stories:
- **$PE$ vs step** — does the basin keep getting deeper, or are we plateauing?
- **$|F|$ vs step** — at $|F| \to 0$ we are at mechanical equilibrium. The slope
  tells you whether ADAM still has gradient signal to act on, or if the residual
  $|F|$ is locked into a saddle direction the optimizer can't access.

The $|F|$ curve is the cleaner stopping signal: $PE$ has a non-zero floor
(line tension + area pressure) but $|F|$ should vanish at any true equilibrium.

In [ ]:
if adam_log:
    its     = np.array([r[0] for r in adam_log])
    pes     = np.array([r[1] for r in adam_log])
    f_max   = np.array([r[2] for r in adam_log])
    f_mean  = np.array([r[3] for r in adam_log])

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # PE — linear and log
    for ax, ylog in zip(axes[0], [False, True]):
        ax.axhline(PE_swell,   color='k', ls='--', label=f'PE_swell={PE_swell:.3e}')
        ax.axhline(PE_physics, color='r', ls='--', label=f'PE_physics={PE_physics:.3e}')
        ax.plot(its, pes, 'b-o', ms=3, label='PE during ADAM')
        ax.set_xlabel('ADAM step')
        ax.set_ylabel('PE')
        if ylog:
            ax.set_yscale('log')
        ax.legend(); ax.grid(alpha=0.3)
        ax.set_title(f"PE convergence (lr={ADAM_LR})  ({'log' if ylog else 'linear'} y)")

    # |F| — linear and log
    for ax, ylog in zip(axes[1], [False, True]):
        ax.plot(its, f_max,  'r-o', ms=3, label='|F|_max')
        ax.plot(its, f_mean, 'g-o', ms=3, label='|F|_mean')
        ax.set_xlabel('ADAM step')
        ax.set_ylabel('|F| per node')
        if ylog:
            ax.set_yscale('log')
        ax.legend(); ax.grid(alpha=0.3)
        ax.set_title(f"Force-balance convergence  ({'log' if ylog else 'linear'} y)")

    plt.tight_layout()
    plt.show()

---
## Section 8 — Regression check: physics restart from ADAM state

Re-enable physical xi-drag and run standard physics for 1000 steps from the ADAM state.
Verifies that ADAM didn't move particles into geometrically invalid positions and that
`sync_from_var` left the state consistent for the integrator.

In [ ]:
s._params['xi_drag_per_p'] = xi_phys     # restore physical drag

t0 = time.time()
s.run(1000, sample_every=1000, verbose=False, record_initial=False)
reg_wall = time.time() - t0
nan_count = int(np.isnan(s._state['x_cm'].numpy()).sum())
print(f'Regression: 1000 physics steps in {reg_wall:.2f}s  '
      f'({reg_wall/1000*1000:.2f} ms/step)  NaN count = {nan_count}')

fig, ax = plt.subplots(figsize=(7, 7))
s.set_color_palette('palette1', seed=1)
s.render(ax=ax)
ax.set_title(f'After 1000 physics steps from ADAM state\n'
             f'NaN={nan_count}  φ={s.phi_outer:.4f}')
plt.show()

---
## Summary

**Phase 8 workflow** (4 new System methods, 1 new TF function):
- `make_position_variable()` — convert `x_all` to a `tf.Variable` for the optimizer
- `eval_forces_at(x_all)` — pure-TF force evaluation, gradient-flowing
- `eval_potential_energy(x_all=None)` — scalar PE diagnostic
- `eval_potential_energy_components(x_all=None)` — per-term PE dict
- `sync_from_var(x_var)` — commit optimizer result back to System state (uses θ=0
  decomposition so $x_{\rm all} = x_{cm} + (X_{\rm ref} + u)$ stays consistent for the integrator)

**Reading the breakdown**: ADAM mostly drops the **contact penalty (`cc`)** term — that's
saddle-escape (rearranging contacts). Line tension barely moves at high φ (mostly
geometry-fixed). Area term stays roughly at the box-imposed floor.

**Reading the $|F|$ curve**: this is the more honest stopping criterion. PE has a non-zero
floor at any compressed state (line tension + area), so absolute PE comparison can be
misleading. $|F|$ tells you how close you are to a true mechanical equilibrium. A
plateau in $|F|$ at a non-zero value means ADAM has hit a saddle it can't escape.

**Use it as IC prep for production runs**: swell → ADAM → physics. The post-ADAM state
is mechanically equilibrated, so the subsequent physics run starts at low KE without
drift artifacts.

**Drop-in optimizer swap**: replace `tf.keras.optimizers.Adam` with `AdamW`, `Lion`,
`RMSProp`, etc. — all work without code changes since we treat the force as the gradient.
Try Lion (sign-only updates) for problems where momentum dominates.

---
## Section 9 — ADAM-as-integrator swell (recommended for IC prep)

**The big idea**: instead of running physics dynamics during compression and then relaxing afterward with ADAM, replace the physics step with an ADAM step while the box compresses. Each iteration:

1. **ADAM step on −F** (move toward local energy minimum)
2. **Box compression on x_var** via the same `(1 + rate·dt)` formula the kernel uses on x_cm
3. **Bookkeeping** (sync_box, PRCM refresh) every K steps

ADAM has no physical inertia, so the system tracks the local minimum throughout compression — never landing in a metastable state with stored kinetic energy. Result: PE stays pinned at the basin floor (~105) the entire swell instead of climbing to ~109.5 and then needing a 5k-step ADAM relaxation to recover.

**Required fix for the rate controller**: the user-validated swell scheme has a `0.99 decay per chunk` rate that geometrically accumulates over many chunks. For long ADAM-swells (N_SWELL ~ 384k means ~3840 chunks), the rate decays to ~5e-17 by oscillation onset, freezing phi short of target. Fix is a **dφ-per-chunk floor** (4e-5): each chunk on a compress half-cycle must advance phi by at least the floor, regardless of what the controller's decay logic produces. Only kicks in when controller would otherwise stall; silently ignored otherwise.

This is the **recommended approach for production IC prep**: drop-in for `s.run` during the swell phase, give a clean equilibrated configuration as output. Tradeoff: ~10× wall time vs physics-swell, but no post-swell relaxation needed and IC quality is dramatically better.

| approach | φ | PE | \|F\|_max | \|F\|_mean |
|---|---|---|---|---|
| Physics-swell + 5k ADAM relax | 0.860 | 105.60 | 0.561 | 0.0149 |
| **ADAM-swell (3×, with dφ floor)** | **0.860** | **105.06** | 0.207 | **0.00325** |

(Cell below runs the ADAM-swell from a fresh System; `N_SWELL_ADAM` is set small for notebook demonstrability — bump to 100k+ for production.)

In [ ]:
# Build a fresh system for the ADAM-swell demo (so it's self-contained;
# safe to run in any order after Section 1 setup).
s2 = System(Lx=18.0, Ly=18.0, periodic_x=True, periodic_y=True,
            dt_factor=0.25, candidacy_kind='prcm', E_candidates=9)
s2.add_particles(ParticleSpec(
    count=P, type='emulsion', gamma=1.0, kappa=0.02,
    N_nodes=N, Oh=OH,
    poly_dist={'type': 'bimodal', 'ratio': 0.5, 'delta': 0.2}))
s2.initialize(phi_target=PHI_INIT, seed=SEED, verbose=False,
              relax_only=False, n_relax_init=200)

DTYPE2 = s2._params['_alpha_tf'].dtype
dt2    = float(s2._params['_dt_tf'].numpy())
xi_phys2 = tf.constant(s2._params['xi_drag_per_p'].numpy(), dtype=DTYPE2)

# Zero ALL damping (ADAM has its own dissipation)
s2._params['xi_drag_per_p']    = tf.zeros_like(xi_phys2)
s2._params['_alpha_tf']        = tf.constant(0.0, dtype=DTYPE2)
s2._params['alpha_damp_per_p'] = tf.zeros([P], dtype=DTYPE2)
s2.beta_rb = 0.0

# ── ADAM-swell parameters ─────────────────────────────────────────────────────
N_SWELL_ADAM   = 50_000        # bump to 100k+ for production-quality IC
BOOK_EVERY     = 100
PRCM_EVERY_AS  = 50
DPHI_FLOOR_PER_CHUNK = 4e-5    # floor on phi advance per chunk (compress only)
N_SWELL_CAP    = int(2.0 * N_SWELL_ADAM)

opt2   = tf.keras.optimizers.Adam(learning_rate=ADAM_LR)
x_var2 = s2.make_position_variable()
theta_np2 = s2._state['theta'].numpy()
s2._cm_mgr.update(x_var2.numpy().mean(axis=1), theta_np2, x_all=x_var2.numpy())

phi_init_actual = s2.phi_outer
target_dphi_dt_init = (PHI_TARGET - phi_init_actual) / (N_SWELL_ADAM * dt2)
rate_init_as = -2.0 * target_dphi_dt_init / (2.0 * phi_init_actual)
s2.box_rate = (rate_init_as, rate_init_as)

oscillation_triggered = False; direction = -1.0
dt_const = tf.constant(dt2, dtype=DTYPE2)
rate_tf  = tf.constant([rate_init_as, rate_init_as], dtype=DTYPE2)

print(f'ADAM-swell: phi {phi_init_actual:.4f} → {PHI_TARGET}  '
      f'N_SWELL_ADAM={N_SWELL_ADAM}  floor={DPHI_FLOOR_PER_CHUNK}')
log_as = []
steps = 0
t0 = time.time()
while s2.phi_outer < PHI_TARGET - 1e-6 and steps < N_SWELL_CAP:
    if steps % BOOK_EVERY == 0:
        phi_now = s2.phi_outer
        n_remain = max(N_SWELL_ADAM - steps, BOOK_EVERY)
        rate_test = -((PHI_TARGET - phi_now) / (n_remain * dt2)) / (2.0 * phi_now)
        cur_rate = s2.box_rate[0]
        if abs(rate_test) < abs(cur_rate) and steps < N_SWELL_ADAM:
            rate = rate_test
        else:
            rate = cur_rate * 0.99
        if (PHI_TARGET - phi_now) <= 0.02 or oscillation_triggered:
            if not oscillation_triggered:
                oscillation_triggered = True
                print(f'  [anneal] oscillation triggered  steps={steps}  Δφ={PHI_TARGET-phi_now:.4f}')
            if direction < 0: rate =  abs(rate) / 2
            else:             rate = -abs(rate) * 2
            direction = -direction
        # dφ-per-chunk floor on compress half-cycles
        rate_floor_compress = -DPHI_FLOOR_PER_CHUNK / (2.0 * phi_now * BOOK_EVERY * dt2)
        if rate < 0 and abs(rate) < abs(rate_floor_compress):
            rate = rate_floor_compress
        s2.box_rate = (rate, rate)
        rate_tf = tf.constant([rate, rate], dtype=DTYPE2)

    # ADAM step
    f = s2.eval_forces_at(x_var2)
    opt2.apply_gradients([(-f, x_var2)])

    # Box compression on x_var: shift each node by Δx_cm = x_cm·rate·dt
    x_cm_v   = tf.reduce_mean(x_var2, axis=1, keepdims=True)
    delta_cm = x_cm_v * rate_tf[None, None, :] * dt_const
    x_var2.assign_add(tf.broadcast_to(delta_cm, x_var2.shape))

    if (steps + 1) % BOOK_EVERY == 0:
        s2.sync_box(BOOK_EVERY)
    if (steps + 1) % PRCM_EVERY_AS == 0:
        x_np = x_var2.numpy()
        if s2._cm_mgr.needs_update(x_np.mean(axis=1), theta_np2):
            s2._cm_mgr.update(x_np.mean(axis=1), theta_np2, x_all=x_np)
    steps += 1
    if steps % (N_SWELL_ADAM // 20) == 0:
        x_now = tf.constant(x_var2.numpy(), dtype=DTYPE2)
        PE_now = s2.eval_potential_energy(x_now)
        f_mag  = tf.norm(s2.eval_forces_at(x_var2), axis=2).numpy()
        log_as.append((steps, s2.phi_outer, PE_now, f_mag.max(), f_mag.mean()))

s2.box_rate = 0.0
swell_wall_as = time.time() - t0
PE_as_end      = s2.eval_potential_energy(tf.constant(x_var2.numpy(), dtype=DTYPE2))
PE_as_end_comp = s2.eval_potential_energy_components(tf.constant(x_var2.numpy(), dtype=DTYPE2))
f_end          = s2.eval_forces_at(x_var2)
f_end_mag      = tf.norm(f_end, axis=2).numpy()
print(f'\nADAM-swell done: {steps} steps in {swell_wall_as:.1f}s  '
      f'({swell_wall_as/steps*1000:.2f} ms/step)')
print(f'phi = {s2.phi_outer:.4f} (target {PHI_TARGET:.4f})  '
      f'PE = {PE_as_end:.4e}  |F|_max = {f_end_mag.max():.3e}  '
      f'|F|_mean = {f_end_mag.mean():.3e}')

# Side-by-side comparison: ADAM-swell vs Physics-swell+ADAM (from Sections 3-4)
print(f'\n  Method                    PE       |F|_max     |F|_mean')
print(f'  ' + '-' * 56)
print(f'  Physics + 5k ADAM   {PE_adam:>10.4e}  {f_final_mag.max():>10.3e}  {f_final_mag.mean():>10.3e}')
print(f'  ADAM-as-integrator  {PE_as_end:>10.4e}  {f_end_mag.max():>10.3e}  {f_end_mag.mean():>10.3e}')

# Render final config
s2.sync_from_var(x_var2)
fig, ax = plt.subplots(figsize=(7, 7))
s2.set_color_palette('palette1', seed=1)
s2.render(ax=ax)
ax.set_title(f'ADAM-as-integrator swell  PE={PE_as_end:.3e}  φ={s2.phi_outer:.4f}')
plt.show()

---
## Section 10 — Push |F|_max lower: FIRE & L-BFGS-on-||F||²

ADAM plateaus at |F|_max ≈ 0.2 because at the contact-cusp boundaries (gap ≈ 0)
the contact penalty has a Hessian step that confuses adaptive-step methods.
Two add-ons that go further:

**FIRE** (Bitzek 2006) — physics-aware: Verlet-like velocity update, kill velocity
on F·v < 0 (i.e., when going uphill), adapt dt and mixing strength based on
consecutive downhill steps. Standard for MD energy minimization. With "tight"
knobs (smaller dt_max, larger N_min) it handles cusps gently.

**L-BFGS-B on ||F||²/2** — minimize the squared force norm directly. Self-consistent
gradient via TF GradientTape on the force kernel. The catch: the kernel's
`internal_forces_tf` has `if g != 0:` (gravity branch); when `g` is a TF tensor
this becomes a `tf.cond` whose gradient breaks XLA on GPU. **Fix**: pass
`g=0.0` as Python float so the branch dies at trace time.

**Best chain in our study**: FIRE_tight 25k → L-BFGS-on-||F||² 14k = **|F|_max ≈ 2.5e-3**
(199× drop from post-swell). This is numerical-precision floor in float64, not
a physical limit — soft-mode ill-conditioning + machine epsilon, not cusps.

In [ ]:
from scipy.optimize import minimize
from src.simulation.tf_sim import (internal_forces_tf, k_reg_forces_tf,
                                     inter_capsule_forces_tf, primitive_forces_tf)

# We'll use the post-swell state from Section 9's ADAM-swell (s2 / x_var2)
# Or, if you ran Section 4, use s / x_var. Pick whichever is in memory:
try:
    sys_q = s2; x_quench = s2._state['x_all'].numpy().copy()
    print('Using s2 (post Section 9 ADAM-swell)')
except NameError:
    sys_q = s;  x_quench = s._state['x_all'].numpy().copy()
    print('Using s (post Section 4 ADAM)')

# Make the force kernel differentiable through GradientTape, with the g=0.0 Python-float fix
def force_diff(x_all, caps, params, prim_data):
    """Pure-tensor force kernel that's gradient-flowing for TF GradientTape.
    Pass g=0.0 (Python float, NOT tensor) to internal_forces_tf so its
    `if g != 0` branch dies at trace time — avoids tf.cond + XLA gradient bug."""
    N_nodes = x_all.shape[1]
    r_c_flat = tf.repeat(params['r_c_per_p'], N_nodes)
    k_c_flat = tf.repeat(params['k_c_per_p'], N_nodes)
    L0_flat  = tf.repeat(params['L0'],        N_nodes)
    box      = params.get('box', None)
    f_contact = inter_capsule_forces_tf(x_all, caps, r_c_flat, k_c_flat, L0_flat, box=box)
    if prim_data is not None:
        t0 = tf.constant(0.0, dtype=x_all.dtype)
        f_contact = f_contact + primitive_forces_tf(x_all, t0, prim_data,
                                                     r_c_flat, k_c_flat, L0_flat, box=box)
    f_elastic = internal_forces_tf(x_all, params, 0.0)   # <-- python float
    f_reg     = k_reg_forces_tf(x_all, params)
    return f_elastic + f_reg + f_contact


# ── FIRE optimizer (Bitzek 2006) as a standalone function ────────────────────
def run_fire(sys_, x_var_, n_steps,
             dt_init=0.005, dt_max=0.03,            # 'tight' settings
             alpha_start=0.05, N_min=20,
             f_inc=1.1, f_dec=0.5, f_alpha=0.99,
             prcm_every=100, sample_every=200, verbose=False):
    """FIRE force-quench. Returns (log, n_resets) where log = [(step,|F|max,|F|mean,PE), ...]."""
    DTYPE = sys_._state['x_all'].dtype
    v_var = tf.Variable(tf.zeros_like(x_var_), trainable=False, dtype=DTYPE)
    TINY  = tf.constant(1e-30, dtype=DTYPE)
    dt = dt_init; alpha = alpha_start; n_pos = 0; n_resets = 0

    theta_np = sys_._state['theta'].numpy()
    sys_._cm_mgr.update(x_var_.numpy().mean(axis=1), theta_np, x_all=x_var_.numpy())

    f0 = sys_.eval_forces_at(x_var_)
    f0_mag = tf.norm(f0, axis=2).numpy()
    PE0 = sys_.eval_potential_energy(tf.constant(x_var_.numpy(), dtype=DTYPE))
    log = [(0, float(f0_mag.max()), float(f0_mag.mean()), float(PE0))]

    for i in range(n_steps):
        f = sys_.eval_forces_at(x_var_)
        # F1: power F·v
        Pn = float(tf.reduce_sum(f * v_var).numpy())
        # F2: mix v toward F̂
        v_norm = tf.sqrt(tf.reduce_sum(v_var * v_var) + TINY)
        f_norm = tf.sqrt(tf.reduce_sum(f * f) + TINY)
        v_var.assign((1.0 - alpha) * v_var + (alpha * v_norm / f_norm) * f)
        # F3/F4: dt/alpha update on continued downhill / reset on uphill
        if Pn > 0.0 and n_pos > N_min:
            dt = min(dt * f_inc, dt_max); alpha = alpha * f_alpha
        if Pn <= 0.0:
            dt = dt * f_dec; v_var.assign(tf.zeros_like(v_var))
            alpha = alpha_start; n_pos = 0; n_resets += 1
        else:
            n_pos += 1
        # F5: Verlet integrate
        v_var.assign_add(dt * f); x_var_.assign_add(dt * v_var)

        if (i + 1) % prcm_every == 0:
            x_np = x_var_.numpy()
            if sys_._cm_mgr.needs_update(x_np.mean(axis=1), theta_np):
                sys_._cm_mgr.update(x_np.mean(axis=1), theta_np, x_all=x_np)
        if (i + 1) % sample_every == 0:
            f_mag = tf.norm(f, axis=2).numpy()
            PE = sys_.eval_potential_energy(tf.constant(x_var_.numpy(), dtype=DTYPE))
            log.append((i + 1, float(f_mag.max()), float(f_mag.mean()), float(PE)))
    return log, n_resets


# ── L-BFGS-B on ||F||²/2 with self-consistent GradientTape gradient ──────────
def run_lbfgs_force_norm(sys_, x_var_, max_iter=5000, prcm_every=50, verbose=False):
    """L-BFGS-B on 0.5*||F||² with self-consistent gradient via GradientTape.
    Returns (log, scipy_result) where log = [(eval, |F|max, |F|mean, L), ...]."""
    DTYPE = sys_._state['x_all'].dtype
    SHAPE = x_var_.shape
    theta_np = sys_._state['theta'].numpy()
    n_evals = [0]; log = []

    def f_and_grad(x_flat):
        n_evals[0] += 1
        x_3d = tf.constant(x_flat.reshape(SHAPE), dtype=DTYPE)
        x_var_.assign(x_3d)
        if n_evals[0] % prcm_every == 0:
            x_np = x_var_.numpy()
            if sys_._cm_mgr.needs_update(x_np.mean(axis=1), theta_np):
                sys_._cm_mgr.update(x_np.mean(axis=1), theta_np, x_all=x_np)
        caps = tf.constant(sys_._cm_mgr.CapCandidates, dtype=tf.int32)
        x_local = tf.Variable(x_3d, trainable=True)
        with tf.GradientTape() as tape:
            F = force_diff(x_local, caps, sys_._params, sys_._prim_data)
            L = 0.5 * tf.reduce_sum(F * F)
        grad = tape.gradient(L, x_local).numpy().reshape(-1)

        F_np  = F.numpy()
        f_mag = np.linalg.norm(F_np, axis=2)
        log.append((n_evals[0], float(f_mag.max()), float(f_mag.mean()), float(L.numpy())))
        if verbose and n_evals[0] % 100 == 0:
            print(f'  eval {n_evals[0]:>5}  L={float(L.numpy()):.4e}  '
                  f'|F|_max={f_mag.max():.3e}  |F|_mean={f_mag.mean():.3e}')
        return float(L.numpy()), grad

    res = minimize(f_and_grad, x_var_.numpy().reshape(-1),
                    method='L-BFGS-B', jac=True,
                    options={'maxiter': max_iter, 'maxfun': max_iter * 10,
                             'gtol': 1e-20, 'ftol': 1e-20, 'disp': False})
    return log, res


# ── Run the chain: FIRE_tight 5000 → L-BFGS 1000 ─────────────────────────────
# (5000 + 1000 is a notebook-demo budget; bump to 25000 + 14000 for production)
x_var_q = sys_q.make_position_variable()
F0 = sys_q.eval_forces_at(x_var_q); F0_mag = tf.norm(F0, axis=2).numpy()
print(f'\nStart: |F|_max={F0_mag.max():.3e}  |F|_mean={F0_mag.mean():.3e}\n')

print('FIRE_tight 5000 steps...')
t0 = time.time()
log_fire, resets = run_fire(sys_q, x_var_q, 5000)
print(f'  done in {time.time()-t0:.1f}s  resets={resets}')
F_after_fire = sys_q.eval_forces_at(x_var_q); F_after_fire_mag = tf.norm(F_after_fire, axis=2).numpy()
print(f'  |F|_max={F_after_fire_mag.max():.3e}  |F|_mean={F_after_fire_mag.mean():.3e}\n')

print('L-BFGS-on-||F||² 1000 iterations...')
t0 = time.time()
log_lbfgs, res = run_lbfgs_force_norm(sys_q, x_var_q, max_iter=1000)
print(f'  done in {time.time()-t0:.1f}s  iters={res.nit}  msg={res.message}')
F_final = sys_q.eval_forces_at(x_var_q); F_final_mag = tf.norm(F_final, axis=2).numpy()
print(f'  |F|_max={F_final_mag.max():.3e}  |F|_mean={F_final_mag.mean():.3e}')

# Convergence plot
fig, ax = plt.subplots(figsize=(11, 5))
arr_f  = np.array(log_fire)
arr_l  = np.array(log_lbfgs)
ax.semilogy(arr_f[:, 0], arr_f[:, 1], 'g-', lw=1.3, label='FIRE_tight |F|_max')
ax.semilogy(arr_f[:, 0], arr_f[:, 2], 'g--', lw=1.0, alpha=0.6, label='FIRE_tight |F|_mean')
shift = arr_f[-1, 0]
ax.semilogy(shift + arr_l[:, 0], arr_l[:, 1], 'b-', lw=1.3, label='L-BFGS |F|_max')
ax.semilogy(shift + arr_l[:, 0], arr_l[:, 2], 'b--', lw=1.0, alpha=0.6, label='L-BFGS |F|_mean')
ax.axhline(F0_mag.max(), color='k', ls=':', alpha=0.5, label=f'start ({F0_mag.max():.2e})')
ax.set_xlabel('step / evaluation'); ax.set_ylabel('|F| per node')
ax.legend(); ax.grid(alpha=0.3, which='both')
ax.set_title(f'FIRE + L-BFGS chain — {F0_mag.max()/F_final_mag.max():.0f}× drop from start')
plt.show()

---
## Section 11 — Landscape diagnostics

Three diagnostics that helped us understand WHY |F|_max plateaus where it does:

1. **Per-node |F| histogram + top stuck nodes** — reveals whether residual force is
   distributed (everything's nearly balanced) or *localized* (a few specific nodes
   on a few specific particles are the bottleneck). In our 16-particle test, the
   top 10 stuck nodes were all on **particle 0** — clue toward "frustrated cluster".

2. **Contact-gap distribution** — counts pairs at |gap| < 5e-4 ("cusp band"). The
   contact penalty has a Hessian step at gap=0; pairs sitting near this boundary
   are where Newton-like methods fail.

3. **Hessian spectrum** via finite-difference Hessian-vector products:
   - λ_max via power iteration on H = −∂F/∂x
   - λ_min via shifted power iteration on (λ_max·I − H)
   - κ = λ_max / λ_min — condition number. **κ ≈ 380** in our converged state →
     moderate, NOT the bottleneck. (For comparison: ill-conditioned linear systems
     have κ > 10⁶. Our 380 means standard solvers should work fine in principle.)

The diagnosis: |F|_max plateau is **soft-mode ill-conditioning + numerical precision**,
not contact cusps. With float128 or a properly preconditioned solver, you could push
|F|_max well below the 2.5e-3 we hit in float64.

In [ ]:
# ── Diagnostic 1: per-node |F| histogram + top stuck nodes ───────────────────
F_arr     = sys_q.eval_forces_at(x_var_q).numpy()
F_arr_mag = np.linalg.norm(F_arr, axis=2)   # (P, N)
P_, N_ = F_arr_mag.shape

print(f'  Per-node |F| distribution ({P_*N_} nodes):')
for q in [50, 75, 90, 95, 99]:
    print(f'    {q}th pct = {np.percentile(F_arr_mag.flatten(), q):.3e}')
print(f'    max     = {F_arr_mag.max():.3e}')

print(f'\n  Top 10 stuck nodes:')
flat_idx = np.argsort(-F_arr_mag.flatten())[:10]
print(f"    {'node':<5}  {'particle':<8}  {'|F|':>10}")
for k in flat_idx:
    p_idx, n_idx = k // N_, k % N_
    print(f'    {n_idx:<5}  {p_idx:<8}  {F_arr_mag[p_idx, n_idx]:>10.3e}')


# ── Diagnostic 2: contact-gap distribution ───────────────────────────────────
caps_np = sys_q._cm_mgr.CapCandidates
x_all_np = x_var_q.numpy()
x_flat   = x_all_np.reshape(P_*N_, 2)
r_c_per_p_np = sys_q._params['r_c_per_p'].numpy()
Lx_, Ly_ = sys_q.Lx, sys_q.Ly

def mi(d):
    d = d.copy()
    d[..., 0] -= Lx_ * np.round(d[..., 0] / Lx_)
    d[..., 1] -= Ly_ * np.round(d[..., 1] / Ly_)
    return d

gaps_arr = []
for k in range(P_*N_):
    pa = k // N_
    a0 = x_flat[k]; a1 = x_flat[(pa*N_) + (k+1) % N_]
    for slot in range(caps_np.shape[1]):
        cand = caps_np[k, slot]
        if cand == 0: continue
        b_idx = cand - 1; pb = b_idx // N_
        if pb == pa: continue
        b0 = x_flat[b_idx]; b1 = x_flat[(pb*N_) + (b_idx+1) % N_]
        shift = mi((b0 - a0)[None]).reshape(2) - (b0 - a0)
        b0_s = b0 + shift; b1_s = b1 + shift
        for sg in [(1.0 - 1.0/np.sqrt(3))/2, (1.0 + 1.0/np.sqrt(3))/2]:
            xq = (1 - sg) * a0 + sg * a1
            ab = b1_s - b0_s; ab2 = np.dot(ab, ab)
            if ab2 < 1e-30: continue
            t_B = np.clip(np.dot(xq - b0_s, ab) / ab2, 0, 1)
            cp = b0_s + t_B * ab
            d = np.linalg.norm(xq - cp)
            r_c_pair = r_c_per_p_np[pa] + r_c_per_p_np[pb]
            gaps_arr.append(d - r_c_pair)
gaps_arr = np.array(gaps_arr)

n_overlap = (gaps_arr < 0).sum()
n_cusp    = (np.abs(gaps_arr) < 5e-4).sum()
print(f'\n  Contact-gap analysis ({len(gaps_arr)} (k,e,gauss) entries):')
print(f'    pairs with gap < 0 (overlapping):     {n_overlap}')
print(f'    pairs with |gap| < 5e-4 (cusp band):  {n_cusp}  ({100*n_cusp/len(gaps_arr):.2f}%)')
print(f'    deepest overlap: {gaps_arr.min():+.3e}')

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(F_arr_mag.flatten(), bins=50, color='b', alpha=0.7)
axes[0].set_yscale('log'); axes[0].grid(alpha=0.3)
axes[0].set_xlabel('|F| per node'); axes[0].set_ylabel('count')
axes[0].set_title(f'Per-node |F| (max={F_arr_mag.max():.2e})')

axes[1].hist(gaps_arr, bins=80, color='g', alpha=0.7)
axes[1].axvline(0, color='r', ls='--', label='gap = 0 (cusp)')
axes[1].set_yscale('log'); axes[1].grid(alpha=0.3); axes[1].legend()
axes[1].set_xlabel('contact gap'); axes[1].set_ylabel('count')
axes[1].set_title(f'Contact gaps  ({n_cusp} cusp-band, {n_overlap} overlapping)')
plt.tight_layout(); plt.show()


# ── Diagnostic 3: Hessian spectrum via finite-difference HVPs ────────────────
NDOF = P_ * N_ * 2
EPS_FD = 1e-6
DTYPE = sys_q._state['x_all'].dtype
x_base_flat = x_all_np.reshape(-1)
SHAPE = (P_, N_, 2)

def F_at(x_3d):
    """Force at given 3-D config; returns 3-D numpy."""
    caps = tf.constant(sys_q._cm_mgr.CapCandidates, dtype=tf.int32)
    return force_diff(tf.constant(x_3d, dtype=DTYPE), caps,
                       sys_q._params, sys_q._prim_data).numpy()

def jvp(v_flat):
    """J·v ≈ [F(x+εv) − F(x−εv)] / (2ε)."""
    x_p = (x_base_flat + EPS_FD * v_flat).reshape(SHAPE)
    x_m = (x_base_flat - EPS_FD * v_flat).reshape(SHAPE)
    return ((F_at(x_p) - F_at(x_m)) / (2 * EPS_FD)).reshape(-1)

def Hv(v_flat):
    """H = −J = ∂²U/∂x²."""
    return -jvp(v_flat)

# λ_max via power iteration
print('\n  Hessian spectrum probe...', flush=True)
np.random.seed(1)
v = np.random.randn(NDOF); v /= np.linalg.norm(v)
lam_prev = 0.0
for k in range(80):
    Hv_ = Hv(v)
    lam = float(np.dot(Hv_, v))
    if k > 5 and abs(lam - lam_prev) < 1e-5 * abs(lam): break
    lam_prev = lam
    n = np.linalg.norm(Hv_)
    if n < 1e-30: break
    v = Hv_ / n
lam_max = lam
print(f'    λ_max ≈ {lam_max:.3e}  ({k+1} iters)')

# λ_min via shifted power
np.random.seed(2)
v = np.random.randn(NDOF); v /= np.linalg.norm(v)
lam_prev = 0.0
for k in range(100):
    Hv_ = Hv(v)
    op_v = lam_max * v - Hv_
    lam_shift = float(np.dot(op_v, v))
    if k > 5 and abs(lam_shift - lam_prev) < 1e-5 * abs(lam_shift): break
    lam_prev = lam_shift
    n = np.linalg.norm(op_v)
    if n < 1e-30: break
    v = op_v / n
lam_min = lam_max - lam_shift
kappa = abs(lam_max / max(abs(lam_min), 1e-30))
print(f'    λ_min ≈ {lam_min:.3e}  ({k+1} iters)')
print(f'    κ(H)  ≈ {kappa:.3e}')
print(f'\n  → κ ~ {int(kappa)} is moderate; first-order plateaus are NUMERICAL,')
print(f'    not from cusps or rank-deficiency.')